# CME Futures: Transaction-Cost Sensitivity

This notebook prices one configuration: the one the case study ships, resolved
across labels from the immutable union of signal, allocation and risk-overlay results, with
its risk overlay carried rather than cleared. It then applies the declared all-in cost grid
to that fixed configuration. Commission and slippage each receive half of the grid value.

One curve, not one per horizon. The selected configuration sits on a single label, so the other
label produces no cost rows, and that is the intended state rather than a missing run.

Cost sensitivity is not a selection stage. Its rows are excluded from the final selection pool.
Contract multipliers, tick sizes, margin rates, front-contract position, roll adjustment, and
product identity remain unchanged across the grid.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：交易成本敏感度

本 Notebook 只對一項設定估算成本：也就是本案例研究最終採用的設定。它從訊號、配置與風險覆蓋層結果的不可變聯集中，跨標籤選出該設定，並保留其風險覆蓋層，而非移除。接著將聲明的全包成本網格套用到這項固定設定。網格中的每個數值，一半分配給佣金，一半分配給滑價。

只有一條曲線，而非每個預測期間各一條。選定的設定只屬於單一標籤，因此其他標籤不會產生成本資料列；這是預期狀態，不代表漏跑。

成本敏感度不是選擇階段；其資料列不納入最終選擇集合。合約乘數、最小跳動單位、保證金比率、近月合約部位、轉倉調整及商品識別資訊，在整個網格中都保持不變。
<!-- ML4T_ZH_END c0000-p000 -->


## Why a strategy is priced rather than costed

Every backtest before this one traded for free. That is not an oversight being corrected
here - it is the only way the earlier comparisons could mean anything. A cost assumption
applied during selection silently picks the strategy that best suits *that assumption*, and
since nobody knows the true figure in advance, the result would be a configuration tuned to a
guess. Selecting frictionlessly and pricing afterwards keeps the two decisions separate and
visible.

So this notebook does not ask "what does the strategy earn after costs?" as though there were
one answer. It asks a more useful question: **at what cost level does the edge disappear?**
The grid is swept because that breakeven point, not any single net Sharpe, is what a reader
can compare against what they would actually pay.

### What the number covers

Each grid value is an all-in per-trade cost in basis points, split evenly between commission
and slippage. Those are different things and only the first is knowable in advance:

- **Commission** is the broker and exchange fee. It is a published number, it is small for
  liquid futures, and it does not depend on what the strategy does.
- **Slippage** is the gap between the price that triggered the decision and the price actually
  received. It depends on order size against available depth, on how quickly the position is
  built, and on whether the trade is demanding liquidity or supplying it. It is the larger and
  far less predictable half, and it is the reason a single figure has to be swept rather than
  assumed.


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 為何要對策略估算成本，而非直接算出單一成本結果

在此之前，每次回測都假設交易免費。這不是到此才要修正的疏漏，而是先前比較能有意義的唯一方式。在選擇期間套用成本假設，會悄悄選出最適合*該假設*的策略；既然沒有人能預先知道真實成本，結果就會是針對猜測調整出的設定。先在無摩擦條件下選擇，之後再估算成本，才能讓兩項決策各自清楚可見。

因此，本 Notebook 不會把「扣除成本後策略賺多少？」當成只有一個答案的問題。它問的是更有用的問題：**成本達到什麼水準時，策略優勢會消失？** 掃描整個網格，是因為讀者能拿來與自身實際成本比較的是損益兩平點，而非任一特定成本下的淨夏普比率。

### 數值包含什麼

每個網格數值都是以基點表示的每筆交易全包成本，平均分配給佣金與滑價。兩者不同，且只有前者能預先知道：

- **佣金**是經紀商與交易所收取的費用。它是公開數字，流動性高的期貨通常收費較低，且不取決於策略的行為。
- **滑價**是觸發決策的價格與實際成交價格之間的差距。它取決於訂單規模相對於可用市場深度的大小、建立部位的速度，以及交易是消耗流動性還是提供流動性。它是較大且更難預測的一半，也因此必須掃描多個數值，而不能直接假定一個數字。
<!-- ML4T_ZH_END c0001-p000 -->

The even split between the two is a convention rather than a measurement, and it is worth
naming as one. Nothing here estimates the commission and slippage components separately, so
the split changes no result: the backtest charges the total. It exists so that the grid value
reads as an all-in figure a reader can compare against their own all-in figure, rather than as
a commission a reader might then add slippage to and double-count.

A flat rate across thirty products is a deliberate simplification, and a reader sizing real
positions should know which way it errs. Depth differs by orders of magnitude across this
universe - the equity index contracts absorb size that would move a livestock or lumber
contract several ticks - so one rate is optimistic for the thin products and pessimistic for
the deep ones. It is the right choice here because the alternative is a per-product model
whose assumptions would be doing more work than the measurement.

### Why turnover is what the cost multiplies

Cost is charged per unit traded, not per unit held, so what a grid value buys is entirely
decided by how much the strategy rebalances. Two configurations with identical gross returns
can sit far apart on these curves if one of them churns.

That gives this stage a second use beyond the headline breakeven. A signal whose edge survives
realistic costs is one whose ranking is stable enough that yesterday's positions are mostly
still today's; a signal that dies at a few basis points was earning its gross return by
trading constantly. The first is a strategy and the second is a description of a data
artifact, and no gross Sharpe distinguishes them.

### Why the risk overlay is carried rather than cleared


<!-- ML4T_ZH_BEGIN c0001-p001 -->
兩者各占一半只是慣例，而非衡量結果，應明確說明。此處沒有分別估計佣金與滑價，因此分配比例不會改變任何結果：回測收取的是總成本。這種分法是為了讓網格數值可讀作全包成本，供讀者與自身全包成本比較，而不會誤認為只是佣金，再加一次滑價而重複計算。

對三十種商品使用單一費率，是刻意簡化；準備建立實際部位的讀者應知道它會往哪個方向出錯。這個商品範圍內的市場深度相差數個數量級：股價指數合約可吸收的交易量，足以讓牲畜或木材合約移動數個跳動單位。因此，單一費率對流動性較薄的商品過於樂觀，對深度較大的商品則過於悲觀。此處仍採用這種方式，因為另一種做法是建立逐商品模型，而模型假設的影響將超過實際衡量的影響。

### 為何成本要乘上周轉量

成本按交易量收取，而非按持有量收取，因此同一網格數值帶來的實際成本，完全取決於策略再平衡的幅度。若其中一個設定頻繁換手，即使兩個設定的未扣成本報酬率相同，在成本曲線上仍可能相距很遠。

因此，除了主要的損益兩平點，本階段還有第二種用途。若訊號的優勢在合理成本下仍能維持，代表其排序夠穩定，昨天的部位今天大多仍會持有；若增加幾個基點成本就失效，表示它靠持續交易賺取未扣成本報酬率。前者是策略，後者只是資料假象的描述，而未扣成本的夏普比率無法區分兩者。

### 為何保留風險覆蓋層，而非移除
<!-- ML4T_ZH_END c0001-p001 -->

The selected configuration is priced with its risk overlay in place, because the overlay changes
the positions and therefore changes what is traded. Pricing a bare signal and then adding an
overlay afterwards would cost a strategy nobody runs. This matters more here than it would
elsewhere: an overlay that reduces position size in volatile periods also reduces turnover in
exactly the periods where slippage is worst, so stripping it out would misstate the cost in a
direction that flatters the strategy.

<!-- ML4T_ZH_BEGIN c0001-p002 -->
估算選定設定的成本時會保留其風險覆蓋層，因為覆蓋層會改變部位，進而改變實際交易。若先對未加覆蓋層的訊號估算成本，事後再加上覆蓋層，算的便是無人實際執行的策略。此處這點尤其重要：若覆蓋層在波動較大時縮減部位，也會恰好在滑價最嚴重時降低周轉量；移除它會錯估成本，且錯估方向會讓策略看起來更有利。
<!-- ML4T_ZH_END c0001-p002 -->


In [1]:
"""Run CME futures transaction-cost sensitivity on fixed configurations."""

import json

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    open_study,
    product_universe_table,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.research.population import supersedes_for_run
from case_studies.utils.strategy_analysis import resolve_solvent_carrier
from case_studies.utils.sweep_config import get_cost_grid_bps

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []

# The cost population is immutable under its name, so a run whose members have moved has to say
# which generation it retires. Anything upstream that changes a backtest identity moves them - a
# corrected label, a changed accounting field, a re-run after a registry reset - and
# `OfficialPopulation.create` refuses to write a different member list under a name that already
# exists. Declared as a literal so that running the committed notebook as it stands recomputes
# the population on record. Empty for a first snapshot.
COST_POPULATION = "cme_futures-cost-validation-v1"
SUPERSEDES_COST_POPULATION: str = ""

## Fixed inputs

There is one configuration to price, and the shared selector below decides which
label it is on. `PREVIEW_LABELS` is still validated rather than ignored: a preview run
that names a label the case study does not declare is a mistake worth stopping, even
though nothing here loops over the set.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 固定輸入

只有一項設定需要估算成本，下方的共用選擇器會決定它屬於哪個標籤。`PREVIEW_LABELS` 仍會受到驗證，而非被忽略：若預覽執行指定案例研究未聲明的標籤，即使此處未對整個集合迴圈處理，也值得停止並報錯。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS:
        raise ValueError("canonical execution cannot declare preview reductions")
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_LABELS")
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
cost_grid = get_cost_grid_bps("cme_futures")
if not cost_grid:
    raise ValueError("the configured cost grid is empty")

# The configuration this case study reports, resolved once by the shared selector rather
# than ranked again here. `resolve_solvent_carrier` goes through
# `resolve_canonical_rank1_lineage`, which ranks across the signal, allocation and
# risk-overlay stages together, re-ranks conformal candidates on exact common timestamp
# support and applies LABEL_RESTRICTIONS, UNIVERSE_RESTRICTIONS and CARRIER_PINS. A plain
# Sharpe ranking beside it does none of those, and where the two disagree this notebook
# prices a strategy the chapter does not report while `19_strategy_analysis` finds no cost
# rows for the configuration it selected. It also refuses a configuration whose equity reached zero,
# whose Sharpe would be computed on a balance that no longer exists.
#
# A per-label loop running a cost grid off the pre-overlay allocation results would price
# something else twice over: there is one strategy, not one per label, and the stage before risk
# management holds the pre-overlay winner rather than the configuration that is actually
# shipped.
carrier = resolve_solvent_carrier("cme_futures")
strategy = json.loads(carrier["spec_json"])["strategy"]
selected_label = carrier["label"]
prediction_hash = carrier["val_prediction_hash"]
print(
    f"Pricing the canonical validation rank-1: {carrier['val_backtest_hash']} "
    f"({carrier['family']}/{carrier['config_name']}) on {selected_label}, "
    f"from the {carrier['val_stage']} stage, validation Sharpe {carrier['val_sharpe']:.3f}, "
    f"max drawdown {carrier['max_drawdown']:.3f}."
)
print(
    f"  It carries {'a' if strategy.get('risk') else 'no'} risk overlay, and that block "
    "travels with it below rather than being cleared."
)

request_rows = []
for total_cost_bps in cost_grid:
    request_rows.append(
        {
            "request_name": f"{carrier['val_backtest_hash']}-cost-{total_cost_bps:g}",
            "prediction_hash": prediction_hash,
            "label": selected_label,
            "signal": strategy["signal"],
            "allocation": strategy.get("allocation"),
            # Carried, not cleared. The configuration being priced is the one that came out
            # of risk management, and dropping its overlay here would price a strategy
            # nobody selected.
            "risk": strategy.get("risk"),
            "costs": {
                "commission_bps": total_cost_bps / 2,
                "slippage_bps": total_cost_bps / 2,
            },
            "chapter": "ch18",
        }
    )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "costs")

Pricing the canonical validation rank-1: 5075cacaecb2 (latent_factors/sdf) on fwd_ret_21d, from the risk_overlay stage, validation Sharpe 1.100, max drawdown -0.203.
  It carries a risk overlay, and that block travels with it below rather than being cleared.


request_name,prediction_hash,label,costs
str,str,str,object
"""5075cacaecb2-cost-0""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 0.0, 'slippage_bps': 0.0}"
"""5075cacaecb2-cost-1""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 0.5, 'slippage_bps': 0.5}"
"""5075cacaecb2-cost-2""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 1.0, 'slippage_bps': 1.0}"
"""5075cacaecb2-cost-3""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 1.5, 'slippage_bps': 1.5}"
"""5075cacaecb2-cost-5""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 2.5, 'slippage_bps': 2.5}"
…,…,…,…
"""5075cacaecb2-cost-10""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 5.0, 'slippage_bps': 5.0}"
"""5075cacaecb2-cost-15""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 7.5, 'slippage_bps': 7.5}"
"""5075cacaecb2-cost-20""","""430e5789167d""","""fwd_ret_21d""","{'commission_bps': 10.0, 'slippage_bps': 10.0}"


## Execute the declared grid

Expected identities are snapshotted before execution. An empty input, failed grid member, missing
sidecar, or incomplete lineage fails the notebook instead of reporting a smaller population.

Failing rather than reporting a smaller population is the important half of that sentence. A
cost curve with a missing point still plots, and it plots as a curve that bends somewhere it
does not really bend - so the one output a reader would take a breakeven from is exactly the
one that hides an incomplete run. There is no partial answer to this question worth having.

The results are written as a named population for the same reason the baseline sweep is, and
`SUPERSEDES_COST_POPULATION` above is how a re-run says which generation it retires. A cost
curve is quoted in the chapter, so the set of backtests behind a quoted number has to stay
recoverable after the numbers move.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行聲明的網格

執行前先擷取預期識別資訊的快照。輸入為空、網格成員失敗、缺少附屬檔案，或資料沿革不完整，都會使 Notebook 失敗，而不是回報較小的群體。

這句話的關鍵是失敗，而非回報較小的群體。少了一個點的成本曲線仍能畫出來，但曲線看似在某處轉彎，實際上卻沒有；讀者用來推算損益兩平點的唯一輸出，恰好會掩蓋執行不完整的問題。這個問題沒有值得保留的部分答案。

結果會寫成具名群體，理由與基準掃描相同；上方的 `SUPERSEDES_COST_POPULATION` 讓重新執行時能指定要汰換哪個版本。本章會引用成本曲線，因此即使數字變動，引用數值背後的回測集合仍必須能夠還原。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name=COST_POPULATION if EXECUTION_TIER == "canonical" else None,
    supersedes=supersedes_for_run(
        study,
        population_name=COST_POPULATION,
        declared=SUPERSEDES_COST_POPULATION or None,
        execution_tier=EXECUTION_TIER,
    ),
)

`source` says whether each member was computed by this run or served from the registry because
an identical identity was already recorded. A re-run of a registered sweep is entirely `reused`
and completes in seconds; without the column that is indistinguishable from having computed
every row.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
`source` 表示每個成員是本次執行新計算的，還是因為相同識別資訊已記錄在登錄資料庫，而直接取用既有結果。重新執行已登錄的掃描時，所有成員都會是 `reused`，幾秒內即可完成；若沒有這一欄，就無法區分它與重新計算了每一列的情況。
<!-- ML4T_ZH_END c0009-p000 -->


In [6]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete,source
str,str,str,str,str,bool,str
"""5075cacaecb2-cost-0""","""fwd_ret_21d""","""430e5789167d""","""84725cfa8e21""","""6d9a93188e50""",true,"""computed"""
"""5075cacaecb2-cost-1""","""fwd_ret_21d""","""430e5789167d""","""a0bb88be98df""","""d7b76db89a56""",true,"""computed"""
"""5075cacaecb2-cost-10""","""fwd_ret_21d""","""430e5789167d""","""0904f5f88e12""","""3d5cb3127b41""",true,"""computed"""
"""5075cacaecb2-cost-15""","""fwd_ret_21d""","""430e5789167d""","""d08c04777c56""","""82e527f40e91""",true,"""computed"""
"""5075cacaecb2-cost-2""","""fwd_ret_21d""","""430e5789167d""","""590439a31f38""","""9ddc531bbd27""",true,"""computed"""
…,…,…,…,…,…,…
"""5075cacaecb2-cost-3""","""fwd_ret_21d""","""430e5789167d""","""0b06bbb3de21""","""6b7d6e7105a6""",true,"""computed"""
"""5075cacaecb2-cost-30""","""fwd_ret_21d""","""430e5789167d""","""d4fe9c3e1fe7""","""5c3b31077a52""",true,"""computed"""
"""5075cacaecb2-cost-5""","""fwd_ret_21d""","""430e5789167d""","""0bf48bd12636""","""847cd70751ea""",true,"""computed"""


`19_strategy_analysis` may describe the cost curve, but these backtests do not participate in
configuration selection.

## Reading the curve

The quantity to take from this notebook is the cost level at which the net Sharpe reaches
zero, not the net Sharpe at any particular level. That breakeven is a property of the
strategy; the figure a given reader pays is a property of their broker, their size and their
execution, and only they know it.

Two failure modes to keep apart when reading it. A strategy that breaks even far above any
plausible cost is genuinely robust to friction. A strategy that breaks even just above the
plausible range is not "profitable but tight" - it is a strategy whose result depends on an
assumption nobody can pin down, and the honest description of it says so rather than quoting
the frictionless number with a footnote.

What this stage cannot tell you is the cost of trading at a size the market notices. Every
figure here is linear in turnover: doubling the position doubles the cost. Real slippage grows
faster than that once an order is large against available depth, so the curve is a lower bound
on what scale costs, and the point where it crosses zero is optimistic for anyone trading
size. Nothing in this case study measures market impact, and reading the breakeven as a
capacity limit would be reading it for something it does not contain.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
`19_strategy_analysis` 可以描述成本曲線，但這些回測不參與設定選擇。

## 解讀曲線

從本 Notebook 應取得的量，是淨夏普比率降至零時的成本水準，而非任一特定成本水準下的淨夏普比率。損益兩平點是策略的特性；個別讀者實際支付的成本，則由其經紀商、交易規模與執行方式決定，只有讀者自己知道。

解讀時須區分兩種失敗模式。若策略的損益兩平點遠高於任何合理成本，表示它確實能承受交易摩擦。若損益兩平點只略高於合理範圍，就不能稱為「有獲利但餘裕有限」：其結果取決於沒有人能準確掌握的假設，應如實描述，而不是只引用無摩擦條件下的數字，再附上註腳。

本階段無法告訴你，以足以影響市場的規模交易會花多少成本。此處每個數值都與周轉量成正比：部位加倍，成本也加倍。但當訂單相對於可用市場深度很大時，實際滑價的成長會超過這個比例。因此，曲線只是擴大規模所需成本的下界；對大規模交易者而言，淨夏普比率降至零的交點偏樂觀。本案例研究沒有衡量市場衝擊，若把損益兩平點解讀為策略容量上限，就是賦予它原本沒有的意義。
<!-- ML4T_ZH_END c0011-p000 -->
